In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# import subprocess
# import sys
# from pathlib import Path

# package_dir = Path.cwd().resolve()
# if package_dir.name == 'examples':
#     package_dir = package_dir.parent
# else:
#     repo_candidate = Path(
#         'packages/visualization/calibrated-explanations-visualization-plotly'
#     ).resolve()
#     if repo_candidate.exists():
#         package_dir = repo_candidate

# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(package_dir)])
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly>=5.18'])

# Plotly local factual simple

`plotly.local.factual_simple` replicates the compact factual figure from the
explainable-ai-hub ExplainPage: one horizontal bar per rule in **payload order**,
coloured by weight sign (indigo positive, red negative), with a transparent
background and a fixed compact layout.

Compared with `plotly.local.factual_bars` it has **no prediction header, no
ranking, and no filtering** — what CE's `get_rules()` returns is what you see.
Because `get_rules()` returns the merged conjunctive rules once conjunctions
have been added, this style is the easiest way to plot rules *and* conjunctions
together.

Key options:
- `uncertainty=True` (CE-native) or `show_uncertainty=True` — draw weight-interval error bars
- rule labels longer than 32 characters are truncated with an ellipsis; the full
  rule text is kept in the artifact

This is a local factual plot, not a global explanation.

In [3]:
import ce_visualization_plotly.plugin as plotly_plugin
from calibrated_explanations import WrapCalibratedExplainer
from sklearn.datasets import make_classification, make_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

plotly_plugin.register_plotly_visualization_components()  # explicit, idempotent registration


## Classification

In [4]:
X_cls, y_cls = make_classification(
    n_samples=600,
    n_features=8,
    n_informative=5,
    n_redundant=0,
    random_state=7,
)
x_proper, x_temp, y_proper, y_temp = train_test_split(
    X_cls,
    y_cls,
    test_size=0.40,
    random_state=7,
    stratify=y_cls,
)
x_cal, X_query, y_cal, y_query = train_test_split(
    x_temp,
    y_temp,
    test_size=0.50,
    random_state=7,
    stratify=y_temp,
)

model = LogisticRegression(max_iter=1000, solver="liblinear", random_state=7)
explainer = WrapCalibratedExplainer(model)
explainer.fit(x_proper, y_proper)
assert explainer.fitted is True

explainer.calibrate(x_cal, y_cal)
assert explainer.calibrated is True

factual = explainer.explain_factual(X_query[:3])

In [5]:
factual[0].plot(style="plotly.local.factual_simple", show=True);

## Uncertainty error bars

CE's `uncertainty=True` maps to `show_uncertainty=True` inside this plugin.
The error bars span the calibrated weight interval `[weight_low, weight_high]`
around each bar, matching the hub's "uncertain bar-plot" mode.

In [6]:
factual[0].plot(
    style="plotly.local.factual_simple",
    show=True,
    uncertainty=True,
);

## Conjunctions

After `add_conjunctions()`, CE's `get_rules()` returns the merged rule set, so
conjunctive rules appear directly in the figure — the reason this figure exists
in the hub, whose server-side `factual_bars` view does not include them.

In [7]:
factual[0].add_conjunctions(max_rule_size=2)
factual[0].plot(style="plotly.local.factual_simple", show=True, uncertainty=True);

## Regression

In [8]:
X_reg, y_reg = make_regression(
    n_samples=600,
    n_features=8,
    n_informative=6,
    noise=8.0,
    random_state=11,
)
x_proper_reg, x_temp_reg, y_proper_reg, y_temp_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.40,
    random_state=11,
)
x_cal_reg, X_query_reg, y_cal_reg, y_query_reg = train_test_split(
    x_temp_reg,
    y_temp_reg,
    test_size=0.50,
    random_state=11,
)

reg_model = RandomForestRegressor(n_estimators=80, random_state=11)
reg_explainer = WrapCalibratedExplainer(reg_model)
reg_explainer.fit(x_proper_reg, y_proper_reg)
assert reg_explainer.fitted is True

reg_explainer.calibrate(x_cal_reg, y_cal_reg)
assert reg_explainer.calibrated is True

reg_factual = reg_explainer.explain_factual(
    X_query_reg[:3],
    low_high_percentiles=(10, 90),
)

In [9]:
reg_factual[0].plot(style="plotly.local.factual_simple", show=True, uncertainty=True);

## HTML export

In [10]:
# Passing path= to .plot() can conflict with CE's internal kwarg handling.
# Use the returned PlotRenderResult to write the figure directly instead:
result = factual[0].plot(style="plotly.local.factual_simple", show=False)
result.figure.write_html("local_factual_simple.html")